In [0]:
spark



In [0]:
import pandas as pd
import numpy as np
import re

In [0]:
df = pd.read_csv("/Volumes/data13/schema1/volume1/census_data_raw (1).csv")

In [0]:
df.head()

,AGE,workclass,education_level,EDUCATION-NUM,marital-status,OCCUPATION,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,event_time,random_flag,source_system
0,39$,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=80K,20/10/24,Y,batch
1,50$,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,NaN,0.0$,0,13,United-States,<=80K,06/04/26,NaN,batch
2,38,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0$,0,40,United-States,<=80K,11/05/23 10:00,NaN,web
3,53$,NaN,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,NaN,0,0,40,United-States,<=80K,08/07/24,NaN,web
4,28$,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0.0k,40.00%,Cuba,NaN,09/12/24,N,batch


In [0]:
print(df.shape)

(49764, 17)


In [0]:
print(df.columns)

Index(['AGE', 'workclass', 'education_level', 'EDUCATION-NUM',
       'marital-status', 'OCCUPATION', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income', 'event_time', 'random_flag', 'source_system'],
      dtype='object')


In [0]:
df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
)

In [0]:
print(df.columns)

Index(['age', 'workclass', 'education_level', 'education_num',
       'marital_status', 'occupation', 'relationship', 'race', 'sex',
       'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
       'income', 'event_time', 'random_flag', 'source_system'],
      dtype='object')


In [0]:
df.isnull().sum()

age                 3533
workclass           3489
education_level     3371
education_num       3403
marital_status      3458
occupation          3448
relationship        3488
race                3483
sex                 3446
capital_gain        3494
capital_loss        3492
hours_per_week      3552
native_country      3407
income              3572
event_time          3603
random_flag        16515
source_system          0
dtype: int64

In [0]:
df['age'] = (
    df['age']
        .astype(str)
        .str.replace(r'ERROR_\w+', '', regex=True)
        .str.replace(r'[^0-9]', '', regex=True)
)

In [0]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')

In [0]:
df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
)

In [0]:
print(df.columns)

Index(['age', 'workclass', 'education_level', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income', 'event_time', 'random_flag', 'source_system'],
      dtype='object')


In [0]:
df.isnull().sum()


age                 3533
workclass           3489
education_level     3371
education-num       3403
marital-status      3458
occupation          3448
relationship        3488
race                3483
sex                 3446
capital-gain        3494
capital-loss        3492
hours-per-week      3552
native-country      3407
income              3572
event_time          3603
random_flag        16515
source_system          0
dtype: int64

In [0]:
df['age'] = (
    df['age']
        .astype(str)
        .str.replace(r'ERROR_\w+', '', regex=True)
        .str.replace(r'[^0-9]', '', regex=True)
)

In [0]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')

In [0]:
df['workclass'] = (
    df['workclass']
        .fillna('Unknown')
        .astype(str)
        .str.strip()
)

In [0]:
cat_cols = df.select_dtypes(include='object').columns

In [0]:
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

In [0]:
df = df.drop_duplicates(
    subset=['age', 'education_level', 'occupation']
)

In [0]:
print(df.shape)

(6844, 17)


In [0]:
def parse_event_time(x):

    formats = [
        "%Y-%m-%d",
        "%d-%m-%Y",
        "%m/%d/%Y"
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            continue

    return pd.NaT

In [0]:
df['event_time_std'] = df['event_time'].apply(parse_event_time)

In [0]:
parse_failures = df['event_time_std'].isna().sum()

print(parse_failures)

6844


In [0]:
income_mapping = {
    '<=50K': '<=80k',
    '<=50K.': '<=80k',
    '>50K': '>80k',
    '>50K.': '>80k'
}

In [0]:
df['income'] = df['income'].replace(income_mapping)

In [0]:
print(df['income'].unique())

['<=80K' 'nan' '>80K' 'ERROR_514' 'ERROR_127' 'ERROR_628' 'ERROR_295'
 'ERROR_918' 'ERROR_590' 'ERROR_774' 'ERROR_506' 'ERROR_175' 'ERROR_744'
 'ERROR_713' 'ERROR_970' 'ERROR_614' 'ERROR_218' 'ERROR_905' 'ERROR_256'
 'ERROR_801' 'ERROR_689' 'ERROR_474' 'ERROR_954']


In [0]:
numeric_cols = [
    'capital_gain',
    'capital_loss',
    'hours_per_week'
]

In [0]:
for col in numeric_cols:

    df[col] = (
        df[col]
            .astype(str)
            .str.replace(r'[$,%]', '', regex=True)
            .str.replace('k', '000')
            .str.replace(',', '')
    )

    df[col] = pd.to_numeric(df[col], errors='coerce')

In [0]:
print(df.isnull().sum())

age                 202
workclass             0
education_level       0
education_num         0
marital_status        0
occupation            0
relationship          0
race                  0
sex                   0
capital_gain        486
capital_loss        556
hours_per_week      507
native_country        0
income                0
event_time            0
random_flag           0
source_system         0
event_time_std     6844
dtype: int64


In [0]:
print(df['age'].isna().sum())

202


In [0]:
print(
    df[
        ~df['income'].isin(['<=80k', '>80k'])
    ].shape[0]
)

6844


In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6844 entries, 0 to 49763
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   age              6642 non-null   float64       
 1   workclass        6844 non-null   object        
 2   education_level  6844 non-null   object        
 3   education_num    6844 non-null   object        
 4   marital_status   6844 non-null   object        
 5   occupation       6844 non-null   object        
 6   relationship     6844 non-null   object        
 7   race             6844 non-null   object        
 8   sex              6844 non-null   object        
 9   capital_gain     6358 non-null   float64       
 10  capital_loss     6288 non-null   float64       
 11  hours_per_week   6337 non-null   float64       
 12  native_country   6844 non-null   object        
 13  income           6844 non-null   object        
 14  event_time       6844 non-null   object     

In [0]:
df.to_csv("census_cleaned.csv", index=False)

In [0]:
df.head(100)

,age,workclass,education_level,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,event_time,random_flag,source_system,event_time_std
0,39.0,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=80K,20/10/24,Y,batch,NaT
1,50.0,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,nan,0.0,0.0,13.0,United-States,<=80K,06/04/26,nan,batch,NaT
2,38.0,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=80K,11/05/23 10:00,nan,web,NaT
3,53.0,Unknown,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,nan,0.0,0.0,40.0,United-States,<=80K,08/07/24,nan,web,NaT
4,28.0,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,nan,09/12/24,N,batch,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367,57.0,Unknown,HS-grad,9,Divorced,Adm-clerical,Unmarried,White,Female,0.0,NaN,40.0,United-States,nan,08/04/25 10:00,nan,crm,NaT
1370,26.0,Private,HS-grad,9,Divorced,Exec-managerial,Not-in-family,White,Female,0.0,0.0,41.0,United-States,<=80K,nan,Y,batch,NaT
1373,27.0,Private,HS-grad,9,Never-married,Handlers-cleaners,Other-relative,Amer-Indian-Eskimo,Male,0.0,0.0,40.0,United-States,<=80K,10/06/25 10:00,nan,web,NaT
1374,28.0,Self-emp-inc,11th,7.00%,Married-civ-spouse,Craft-repair,nan,White,Male,0.0,0.0,50.0,nan,>80K,01/10/24,nan,batch,NaT


In [0]:
df.head(9
        )

,age,workclass,education_level,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,event_time,random_flag,source_system,event_time_std
0,39.0,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=80K,20/10/24,Y,batch,NaT
1,50.0,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,nan,0.0,0.0,13.0,United-States,<=80K,06/04/26,nan,batch,NaT
2,38.0,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=80K,11/05/23,nan,web,NaT
3,53.0,Unknown,Under Graduate,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,nan,0.0,0.0,40.0,United-States,<=80K,08/07/24,nan,web,NaT
4,28.0,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,nan,09/12/24,N,batch,NaT
5,37.0,Private,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0.0,0.0,40.0,United-States,<=80K,26/09/25,nan,app,NaT
6,49.0,Private,Under Graduate,5,Married-spouse-absent,Other-service,nan,Black,Female,0.0,0.0,16.0,Jamaica,<=80K,02/05/24,N,web,NaT
7,52.0,Unknown,HS-grad,9.00,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,45.0,United-States,nan,20/03/26,N,app,NaT
8,31.0,Private,nan,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084.0,0.0,50.0,United-States,>80K,03/02/24,N,web,NaT


In [0]:
print("census_cleaned_v2.csv exported successfully!")

census_cleaned_v2.csv exported successfully!


In [0]:
df.head(100)

,age,workclass,education_level,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,event_time,random_flag,source_system,event_time_std
0,39.0,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=80K,20/10/24,Y,batch,NaT
1,50.0,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,nan,0.0,0.0,13.0,United-States,<=80K,06/04/26,nan,batch,NaT
2,38.0,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=80K,11/05/23,nan,web,NaT
3,53.0,Unknown,Under Graduate,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,nan,0.0,0.0,40.0,United-States,<=80K,08/07/24,nan,web,NaT
4,28.0,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,nan,09/12/24,N,batch,NaT
5,37.0,Private,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0.0,0.0,40.0,United-States,<=80K,26/09/25,nan,app,NaT
6,49.0,Private,Under Graduate,5,Married-spouse-absent,Other-service,nan,Black,Female,0.0,0.0,16.0,Jamaica,<=80K,02/05/24,N,web,NaT
7,52.0,Unknown,HS-grad,9.00,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,45.0,United-States,nan,20/03/26,N,app,NaT
8,31.0,Private,nan,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084.0,0.0,50.0,United-States,>80K,03/02/24,N,web,NaT
9,42.0,Private,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178.0,0.0,40.0,United-States,nan,09/03/24,nan,web,NaT


In [0]:
df.to_csv(
    r"./census_cleaned_v4.csv"
)

In [0]:
import os
from datetime import datetime

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

model_columns = [
    "age",
    "workclass",
    "education_level",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income",
]

model_pdf = df.copy()
model_pdf.columns = [str(col).strip().lower().replace(" ", "_") for col in model_pdf.columns]

object_cols = model_pdf.select_dtypes(include="object").columns
for col in object_cols:
    model_pdf[col] = model_pdf[col].astype(str).str.strip()
    model_pdf[col] = model_pdf[col].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA, "null": pd.NA})

for col in ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]:
    model_pdf[col] = pd.to_numeric(model_pdf[col], errors="coerce")

income_clean = model_pdf["income"].astype("string").str.strip().str.lower()
income_map = {
    "<=50k": "<=80K",
    "<=50k.": "<=80K",
    "<=80k": "<=80K",
    "<=80k.": "<=80K",
    ">50k": ">80K",
    ">50k.": ">80K",
    ">80k": ">80K",
    ">80k.": ">80K",
}
model_pdf["income"] = income_clean.map(income_map)
model_pdf = model_pdf[model_pdf["income"].notna()].copy()

categorical_columns = [
    "workclass",
    "education_level",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country",
]
numeric_columns = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]

for col in categorical_columns:
    model_pdf[col] = model_pdf[col].fillna("Unknown")

model_pdf = model_pdf[model_columns].copy()
X = model_pdf.drop(columns=["income"])
y = model_pdf["income"]

class_counts = y.value_counts(dropna=False).rename_axis("income").reset_index(name="rows")
print(f"Prepared {len(model_pdf)} labeled rows for modeling")
display(class_counts)
display(model_pdf.head(5))

Prepared 6357 labeled rows for modeling


income,rows
<=80K,5006
>80K,1351


age,workclass,education_level,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
39.0,State-gov,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=80K
50.0,Self-emp-not-inc,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Unknown,0.0,0.0,13.0,United-States,<=80K
38.0,Private,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=80K
53.0,Unknown,Under Graduate,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Unknown,0.0,0.0,40.0,United-States,<=80K
37.0,Private,Masters,14.0,Married-civ-spouse,Exec-managerial,Wife,White,Female,0.0,0.0,40.0,United-States,<=80K


In [0]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print(f"Training rows: {len(X_train)}")
print(f"Validation rows: {len(X_valid)}")
print(f"Test rows: {len(X_test)}")

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_columns,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_columns,
        ),
    ]
)

models_to_train = {
    "logistic_regression": LogisticRegression(max_iter=400),
    "random_forest": RandomForestClassifier(n_estimators=250, max_depth=12, random_state=42, n_jobs=-1),
}

results = []
trained_models = {}
for model_name, estimator in models_to_train.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    validation_predictions = pipeline.predict(X_valid)

    accuracy = accuracy_score(y_valid, validation_predictions)
    f1 = f1_score(y_valid, validation_predictions, pos_label=">80K")

    trained_models[model_name] = pipeline
    results.append({"model_name": model_name, "accuracy": accuracy, "f1": f1})

results_pdf = pd.DataFrame(results).sort_values(["f1", "accuracy"], ascending=False).reset_index(drop=True)
best_model_name = results_pdf.loc[0, "model_name"]
best_model = trained_models[best_model_name]

print(f"Best validation model: {best_model_name}")
display(results_pdf)

Training rows: 4449
Validation rows: 954
Test rows: 954


/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Best validation model: random_forest


model_name,accuracy,f1
random_forest,0.8616352201257862,0.5822784810126582
logistic_regression,0.8480083857442348,0.5821325648414986


In [0]:
test_predictions = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
test_f1 = f1_score(y_test, test_predictions, pos_label=">80K")

run_timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
model_output_dir = f"/Volumes/data13/schema1/volume1/models/census_income_{best_model_name}_{run_timestamp}"
os.makedirs(model_output_dir, exist_ok=True)
model_output_path = os.path.join(model_output_dir, "model.joblib")

sample_input_pdf = X_train.head(5).copy()
sample_output_pdf = pd.DataFrame({"prediction": best_model.predict(sample_input_pdf)})
signature = infer_signature(sample_input_pdf, sample_output_pdf)

mlflow.set_experiment("/Users/shreyashdongare752@gmail.com/work13/census_income_models")
with mlflow.start_run(run_name=f"census_{best_model_name}_{run_timestamp}") as run:
    mlflow.log_param("best_model_name", best_model_name)
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("validation_rows", len(X_valid))
    mlflow.log_param("test_rows", len(X_test))
    mlflow.log_metric("validation_accuracy", float(results_pdf.loc[results_pdf["model_name"] == best_model_name, "accuracy"].iloc[0]))
    mlflow.log_metric("validation_f1", float(results_pdf.loc[results_pdf["model_name"] == best_model_name, "f1"].iloc[0]))
    mlflow.log_metric("test_accuracy", float(test_accuracy))
    mlflow.log_metric("test_f1", float(test_f1))
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        signature=signature,
        input_example=sample_input_pdf,
    )
    run_id = run.info.run_id

joblib.dump(best_model, model_output_path)

summary_pdf = pd.DataFrame(
    [
        {
            "best_model_name": best_model_name,
            "test_accuracy": test_accuracy,
            "test_f1": test_f1,
            "mlflow_run_id": run_id,
            "model_output_path": model_output_path,
        }
    ]
)
prediction_preview = pd.DataFrame({
    "actual_income": y_test.head(5).reset_index(drop=True),
    "predicted_income": pd.Series(test_predictions[:5]),
})

display(summary_pdf)
display(prediction_preview)

/home/spark-d8a5cc50-c065-4f8a-9210-57/.ipykernel/1921/command-8695220905256975-3105655808:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
2026/05/13 08:17:49 INFO mlflow.tracking.fluent: Experiment with name '/Users/shreyashdongare752@gmail.com/work13/census_income_models' does not exist. Creating a new experiment.
2026/05/13 08:17:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-9c5a1bd6-00de.cloud.databricks.com/ml/experiments/1907102675776282/models/m-57670063d85a4e91a00e07440e4f14aa?o=7474649027157496


best_model_name,test_accuracy,test_f1,mlflow_run_id,model_output_path
random_forest,0.8658280922431866,0.6257309941520468,ceee54e0cf7f43789bc06f560e85b0f8,/Volumes/data13/schema1/volume1/models/census_income_random_forest_20260513_081747/model.joblib


actual_income,predicted_income
<=80K,<=80K
<=80K,<=80K
<=80K,<=80K
<=80K,<=80K
>80K,<=80K


In [0]:
inference_input_path = "/Workspace/Users/shreyashdongare752@gmail.com/work13/test/Inference_test_file.csv"
submission_output_path = "/Workspace/Users/shreyashdongare752@gmail.com/work13/submission.csv"
best_model_copy_path = "/Workspace/Users/shreyashdongare752@gmail.com/work13/best_model.joblib"

inference_pdf = pd.read_csv(inference_input_path)
inference_pdf.columns = [
    str(col).strip().lower().replace("-", "_").replace(" ", "_")
    for col in inference_pdf.columns
]

for col in numeric_columns:
    if col in inference_pdf.columns:
        inference_pdf[col] = pd.to_numeric(inference_pdf[col], errors="coerce")

for col in categorical_columns:
    if col in inference_pdf.columns:
        inference_pdf[col] = (
            inference_pdf[col]
            .astype(str)
            .str.strip()
            .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA, "null": pd.NA})
            .fillna("Unknown")
        )

inference_features = inference_pdf.reindex(columns=X.columns)
joblib.dump(best_model, best_model_copy_path)

print(f"Inference rows: {len(inference_features)}")
print(f"Saved best model copy to: {best_model_copy_path}")
display(inference_features.head(5))

Inference rows: 17077
Saved best model copy to: /Workspace/Users/shreyashdongare752@gmail.com/work13/best_model.joblib


age,workclass,education_level,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
44,private,11th,7.0,separated,other-service,unmarried,black,female,0.0,0.0,60.0,united-states
35,private,hs-grad,9.0,married-civ-spouse,craft-repair,husband,white,male,7298.0,0.0,40.0,united-states
35,private,hs-grad,9.0,married-civ-spouse,other-service,other-relative,black,male,0.0,0.0,40.0,united-states
30,private,hs-grad,9.0,married-civ-spouse,transport-moving,husband,white,male,99999.0,0.0,40.0,united-states
62,self-emp-inc,bachelors,13.0,divorced,sales,not-in-family,white,male,99999.0,0.0,40.0,united-states


In [0]:
raw_predictions = best_model.predict(inference_features)
normalized_predictions = pd.Series(raw_predictions).replace({"<=80K": "<=80k", ">80K": ">80k"})

submission_df = pd.DataFrame({
    "id": range(1, len(normalized_predictions) + 1),
    "income": normalized_predictions,
})

submission_df.to_csv(submission_output_path, index=False)

print(f"Saved submission to: {submission_output_path}")
display(submission_df.head(10))

Saved submission to: /Workspace/Users/shreyashdongare752@gmail.com/work13/submission.csv


id,income
1,<=80k
2,>80k
3,<=80k
4,>80k
5,>80k
6,<=80k
7,<=80k
8,>80k
9,<=80k
10,<=80k
